# TRINETRA Kaggle runner
Attach **two** inputs: `happyparsi/trinetra-dataset` and the code dataset (`TRINETRA/` folder from `trinetra_code.zip`).
Set `ACCOUNT` and `PHASE`, enable **GPU T4 x2** and **Internet**, then *Run All*.
Numbers produced here are OUR RESULTS on a prototype subset; never compare them directly with full-benchmark papers.

In [ ]:
ACCOUNT = "A"        # A = fire CNN + colour baseline + video | B = zero-shot | C = deepfake | D = EO heads + geo | F = fusion + RL (after A-C) | P = 7 physics models (tests + real-data hindcast, GLOF, RL env, Geo-PINN) | ALL = everything in one session
PHASE   = "phase2"   # "phase2" = fast (1 seed, few epochs) | "final" = 3 seeds, full epochs
DATASET = "/kaggle/input/trinetra-dataset"
READY   = "/kaggle/input/trinetra-ready-v1/npz"   # used if a teammate already published converted NPZs
GITHUB  = ""         # optional fallback: https URL of a repo that contains TRINETRA/

In [ ]:
import os, subprocess, shutil, glob, pathlib
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode: print(f"[exit {r.returncode}]")
    return r.returncode
# 1) locate code
hits = glob.glob("/kaggle/input/**/trinetra/kaggle/run_hackathon.sh", recursive=True)
CODE = "/kaggle/working/TRINETRA"
if os.path.isdir(CODE + "/trinetra/kaggle"):
    pass  # already unpacked (embedded package)
elif hits:
    src = str(pathlib.Path(hits[0]).parents[2]); shutil.copytree(src, CODE, dirs_exist_ok=True)
elif GITHUB:
    sh(f"git clone --depth 1 {GITHUB} /kaggle/working/repo && cp -r /kaggle/working/repo/TRINETRA {CODE}")
else:
    raise SystemExit("Attach the TRINETRA code dataset (or set GITHUB)")
print("code at", CODE)

In [ ]:
EPOCHS, SEEDS = ("5", "0") if PHASE == "phase2" else ("15", "0 1 2")
os.environ.update(OUT="/kaggle/working/out", EPOCHS=EPOCHS, SEEDS=SEEDS, PY="python",
                  KAGGLE_INPUT=DATASET, DFIRE_N="3000" if PHASE == "phase2" else "4000",
                  CIFAKE_N="5000" if PHASE == "phase2" else "10000")
sh("pip -q install timm tifffile imagecodecs h5py" + (" open_clip_torch" if ACCOUNT in ("B", "ALL") else ""))
sh("nvidia-smi --query-gpu=name,memory.total --format=csv || true")

In [ ]:
# 2) data: find the attached team dataset (Add Input -> happyparsi/trinetra-dataset; it is MOUNTED, not downloaded)
if not os.path.isdir(DATASET):
    cands = [d for d in glob.glob("/kaggle/input/**/", recursive=False) if "trinetra-dataset" in d.lower()] \
         + [d for d in glob.glob("/kaggle/input/*/*/*/") if "trinetra-dataset" in d.lower()]
    if cands:
        DATASET = cands[0].rstrip("/")
    else:
        print("Attached inputs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")
        raise SystemExit("DATASET NOT ATTACHED: right panel -> Add Input -> search happyparsi/trinetra-dataset -> Add. "
                         "It is mounted read-only (no 61 GB download), then Run All again.")
print("dataset at", DATASET); sh(f"ls {DATASET} | head -50")
os.environ["KAGGLE_INPUT"] = DATASET
if ACCOUNT == "P":
    os.environ["DATA"] = "/kaggle/working/npz"; print("physics run: image conversion not needed")
elif os.path.isdir(READY) and glob.glob(READY + "/*.npz"):
    os.environ["DATA"] = READY
else:
    os.environ["DATA"] = "/kaggle/working/npz"
    sh(f"bash {CODE}/trinetra/kaggle/prepare_kaggle_training_data.sh {DATASET} 2>&1 | tee /kaggle/working/prepare.log")
sh("ls -la $DATA; cat /kaggle/working/trinetra_stage/ready_pack_report.json 2>/dev/null | head -c 4000")

In [ ]:
# 3) train / evaluate for this account; results are collected after every stage so a timeout keeps partial outputs
R = f"bash {CODE}/trinetra/kaggle/run_hackathon.sh"
plans = {"A": ["fire", "video"], "B": ["zeroshot"], "C": ["deepfake"], "D": ["eo-train", "geo"], "F": ["fuse"], "P": ["physics"],
         "ALL": ["fire", "deepfake", "zeroshot", "fuse", "video", "eo-train", "physics"]}
status = {}
for stage in plans[ACCOUNT]:
    status[stage] = sh(f"set -o pipefail; {R} {stage} 2>&1 | tee /kaggle/working/{stage}.log")
    sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results > /dev/null 2>&1")
print("stage exit codes:", status)
open("/kaggle/working/stage_status.json", "w").write(__import__("json").dumps(status, indent=2))

In [ ]:
# 4) collect + package (download out.zip or publish /kaggle/working/out as a private dataset version)
sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results")
sh("cp /kaggle/working/*.log /kaggle/working/stage_status.json /kaggle/working/out/ 2>/dev/null; cp -r /kaggle/working/trinetra_stage/*.json /kaggle/working/out/ 2>/dev/null")
shutil.make_archive(f"/kaggle/working/out_{ACCOUNT}_{PHASE}", "zip", "/kaggle/working/out")
sh("cat /kaggle/working/out/results/RESULTS.md 2>/dev/null | head -80")